In [14]:
pip install openai

Note: you may need to restart the kernel to use updated packages.


In [15]:
pip install langchain langchain-openai

Note: you may need to restart the kernel to use updated packages.


In [16]:
# Importar bibliotecas necesarias
from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.chat_history import InMemoryChatMessageHistory
from langchain_core.runnables.history import RunnableWithMessageHistory
import os
import time
from dotenv import load_dotenv

# Cargar variables de entorno desde .env
load_dotenv()

print("Bibliotecas importadas correctamente para streaming")

Bibliotecas importadas correctamente para streaming


In [17]:
# Configuración del modelo con streaming habilitado
try:
    llm = ChatOpenAI(
        base_url=os.getenv("OPENAI_BASE_URL"),
        api_key=os.getenv("GITHUB_TOKEN"),
        model="gpt-4o",  # Modelo disponible en GitHub Models
        streaming=True,  # ¡Importante: habilitar streaming!
        temperature=0.7
    )
    
    print("✓ Modelo configurado con streaming habilitado")
    print(f"Modelo: {llm.model_name}")
    print(f"Streaming: {llm.streaming}")
    
except Exception as e:
    print(f"✗ Error en configuración: {e}")
    print("Verifica las variables de entorno")

✓ Modelo configurado con streaming habilitado
Modelo: gpt-4o
Streaming: True


In [21]:
prompt = ChatPromptTemplate.from_messages([
    ("system", "Eres un exeperto en turismo de Chile."),
    MessagesPlaceholder(variable_name="chat_history"),
    ("human", "{input}")
])

# Cadena = prompt + modelo
chain = prompt | llm

# Almacén de memorias por sesión
store = {}

def get_session_history(session_id: str):
    """Devuelve (o crea) el historial completo para la sesión."""
    if session_id not in store:
        store[session_id] = InMemoryChatMessageHistory()
    return store[session_id]

# Envolver con RunnableWithMessageHistory
conversation = RunnableWithMessageHistory(
    chain,
    get_session_history,
    input_messages_key="input",
    history_messages_key="chat_history"
)

def ejemplo_buffer_memory():
    print("=== CONVERSATIONBUFFERMEMORY ===")
    print("Mantiene todo el historial de conversación\n")
    
    session_id = "demo_session"

    try:
        # Primera interacción
        print("1. Primera pregunta:")
        response1 = conversation.invoke(
            {"input": "Mi nombre es Matias y soy turista, quiero ir a visitar la región de Los Lagos."},
            config={"configurable": {"session_id": session_id}}
        )
        print(f"Respuesta: {response1.content}\n")

        # Segunda interacción
        print("2. Segunda pregunta:")
        response2 = conversation.invoke(
            {"input": "¿Cuál es mi nombre y que soy?"},
            config={"configurable": {"session_id": session_id}}
        )
        print(f"Respuesta: {response2.content}\n")

        # Tercera interacción
        print("3. Tercera pregunta:")
        response3 = conversation.invoke(
            {"input": "¿Qué región de Chile quiero visitar?"},
            config={"configurable": {"session_id": session_id}}
        )
        print(f"Respuesta: {response3.content}\n")

        # Mostrar historial
        print("=== CONTENIDO DE LA MEMORIA ===")
        history = store[session_id].messages
        for i, msg in enumerate(history, 1):
            print(f"{i}. {msg.type}: {msg.content}")

    except Exception as e:
        print(f"Error: {e}")

# Ejecutar
ejemplo_buffer_memory()

=== CONVERSATIONBUFFERMEMORY ===
Mantiene todo el historial de conversación

1. Primera pregunta:
Respuesta: ¡Hola Matías! Bienvenido a Chile y qué excelente elección visitar la Región de Los Lagos. Este lugar es uno de los destinos más hermosos y turísticos del país, lleno de paisajes espectaculares, cultura rica y una deliciosa gastronomía. A continuación, te daré una guía para que disfrutes al máximo tu visita.

### **1. Principales ciudades y destinos en la Región de Los Lagos:**

#### **Puerto Montt**
- Es la capital de la región y el punto de entrada para muchos turistas. Desde aquí puedes visitar el Mercado Angelmó, famoso por sus productos del mar, artesanías y comida típica.
- Pasea por su costanera y disfruta de las vistas al océano y al Volcán Calbuco.

#### **Puerto Varas**
- A solo 20 minutos de Puerto Montt, esta ciudad es conocida por su arquitectura de influencia alemana y su hermosa ubicación frente al Lago Llanquihue.
- Visita la Iglesia del Sagrado Corazón, uno de su

In [ ]:
# --- IMPORTACIONES ---
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.chat_history import InMemoryChatMessageHistory
from langchain_core.runnables.history import RunnableWithMessageHistory
from langchain_core.callbacks.streaming_stdout import StreamingStdOutCallbackHandler
import os

# --- CONFIGURACIÓN DEL MODELO GPT‑4o CON STREAMING ---
stream_handler = StreamingStdOutCallbackHandler()

llm = ChatOpenAI(
    base_url=os.getenv("OPENAI_BASE_URL"),
    api_key=os.getenv("GITHUB_TOKEN"),
    model="gpt-4o",
    streaming=True,
    callbacks=[stream_handler],
    temperature=0.7
)

# --- MEMORIA POR SESIÓN ---
history_store = {}

def get_session_history(session_id):
    if session_id not in history_store:
        history_store[session_id] = InMemoryChatMessageHistory()
    return history_store[session_id]

# --- PROMPT DEL CHATBOT TURÍSTICO ---
prompt = ChatPromptTemplate.from_messages([
    ("system", "Eres un asistente turístico de Chile, amable y experto en destinos nacionales."),
    MessagesPlaceholder(variable_name="chat_history"),
    ("human", "{input}")
])

# --- CREAR LA CADENA CON MEMORIA Y STREAMING ---
conversation = RunnableWithMessageHistory(
    prompt | llm,
    get_session_history,
    input_messages_key="input",
    history_messages_key="chat_history"
)

# --- INTERACCIONES ---
session_id = "matias_turismo"

print("1. Primera pregunta:")
response1 = conversation.invoke(
    {"input": "Mi nombre es Matías y soy turista, quiero visitar la región de Los Lagos"},
    config={"configurable": {"session_id": session_id}}
)
print(f"\nRespuesta: {response1.content}\n")

print("2. Segunda pregunta:")
response2 = conversation.invoke(
    {"input": "¿Cuál es mi nombre y qué soy?"},
    config={"configurable": {"session_id": session_id}}
)
print(f"\nRespuesta: {response2.content}\n")

print("3. Tercera pregunta:")
response3 = conversation.invoke(
    {"input": "¿Qué región de Chile quiero visitar?"},
    config={"configurable": {"session_id": session_id}}
)
print(f"\nRespuesta: {response3.content}\n")

# --- MOSTRAR HISTORIAL DE MEMORIA ---
print("CONTENIDO DE LA MEMORIA:\n")
for msg in history_store[session_id].messages:
    print(msg)

1. Primera pregunta:


ValueError: Invalid input type <class 'dict'>. Must be a PromptValue, str, or list of BaseMessages.